# G1 Humanoid Mocap Tracking with MPPI

This notebook walks through:
1. Installing `mpc-warp` and its dependencies
2. Instantiating `G1MocapTracking` (downloads a walking reference automatically)
3. Running MPPI with `WarpMPPISolver` (GPU via CUDA / CPU Warp kernels)
4. Live visualization in the browser with `mjviser` + `MppiPanel` + orange reference dot cloud
5. Plotting the cost breakdown and pelvis height inline

The task tracks a walking reference from the [LocoMuJoCo](https://huggingface.co/datasets/robfiras/loco-mujoco-datasets) dataset on the Unitree G1 23-DOF humanoid model.

> **Google Colab GPU**: Runtime → Change runtime type → **T4 GPU** before running.
> The solver automatically picks `cuda:0` when a GPU is present.

## 1 · Install dependencies

One cell installs `mpc-warp` from GitHub (pulls in MuJoCo, mujoco-warp, mjviser, etc. from `pyproject.toml`) plus `matplotlib` for the cost plots at the end.

In [ ]:
!pip install -q uv

!uv pip install -q --reinstall \
    "git+https://github.com/Vittorio-Caggiano/mpc-warp.git@main" \
    matplotlib

# On Colab: Runtime → Restart session after this cell, then re-run from cell 2.
print("Install complete.")

## 2 · Verify GPU availability

In [ ]:
import warp as wp

wp.init()
device = wp.get_device().alias
print("Warp device:", device)

## 3 · Create the task and environment

`G1MocapTracking` pairs the Unitree G1 model with a reference-tracking cost. On construction it downloads a walking motion-capture clip and precomputes reference body positions and orientations for every frame.

The running cost is a normalised sum of four exponential-kernel terms:

| Term | What it penalises |
|---|---|
| `body_position` | World-frame body CoM positions (30 bodies) |
| `body_orientation` | World-frame body quaternions (30 bodies) |
| `configuration` | Joint positions (qpos) |
| `velocity` | Joint velocities (qvel) |

Each term lies in `[0, 1]` via `1 − exp(−squared_error)`. Call `task.advance()` after each `env.step()` to keep the reference in sync.

In [ ]:
from mpc_warp.envs.mujoco_env import MujocoTaskEnv
from mpc_warp.tasks.g1_mocap import G1MocapTracking

# Downloads ~15 MB reference file on first run (cached to ~/.cache/huggingface)
task = G1MocapTracking()
env = MujocoTaskEnv(task)
env.reset()

print(f"nq={task.mj_model.nq}  nv={task.mj_model.nv}  nu={task.mj_model.nu}")
print(f"Reference: {task._n_frames} frames at {task._ref_fps:.0f} Hz  ({task._n_frames / task._ref_fps:.1f} s)")
print(f"Bodies tracked: {task._nbody_tracked}")
print(
    f"Cost weights — body_pos={task._w_xpos:.4f}  body_ori={task._w_xquat:.4f}  "
    f"config={task._w_qpos:.4f}  vel={task._w_qvel:.4f}"
)
print(f"Pelvis height at frame 0: {env.data.qpos[2]:.3f} m")

## 4 · Create the MPPI solver

`WarpMPPISolver` allocates `num_samples` parallel MuJoCo worlds on the Warp device.
Each `solver.command(env.data)` call:
1. Uploads the current state to all worlds
2. Rolls out `horizon` steps in parallel
3. Weights samples by `exp(−cost / temperature)`
4. Returns the first action of the updated nominal trajectory

In [ ]:
from mpc_warp.solvers.mppi_warp import WarpMPPIConfig, WarpMPPISolver

num_samples = 64 if device == "cpu" else 256

cfg = WarpMPPIConfig(
    horizon=32,
    num_samples=num_samples,
    noise_sigma=0.3,
    temperature=0.05,
    nominal_return=0.0,
)
solver = WarpMPPISolver(task, cfg, seed=0)
print(f"Solver device: {solver.device}  |  num_samples: {num_samples}")
print("Note: first command() call compiles Warp kernels — expect ~1 min on a cold Colab.")

## 5 · Launch the mjviser browser viewer

`viser.ViserServer` starts a WebSocket server (default port 8080).  
Open the printed URL in a browser to see the 3-D physics scene, the MPPI
diagnostic panels (cost history, ESS, action bars), and an orange reference dot
cloud showing the target body positions for each frame.

> **Colab**: the cell embeds the viewer inline via an `IFrame`.  
> **Re-running this cell** stops the previous server automatically before starting a new one.

In [ ]:
import mujoco
import numpy as np
import viser
from mjviser import ViserMujocoScene

from mpc_warp.viz.mjviser_panel import MppiPanel

# Stop any server left over from a previous run of this cell.
_prev_server = globals().get("server")
if _prev_server is not None:
    _prev_server.stop()

server = viser.ViserServer()  # listens on :8080 by default

act_names = [mujoco.mj_id2name(task.mj_model, mujoco.mjtObj.mjOBJ_ACTUATOR, i) or f"a{i}" for i in range(task.mj_model.nu)]
panel = MppiPanel(server, task.mj_model.nu, act_names)
scene = ViserMujocoScene(server, task.mj_model, num_envs=1)

# Add orange reference dot cloud for mocap body positions
n_ref_pts = task.ref_xpos_viz[0].shape[0]
ref_cloud = server.scene.add_point_cloud(
    "mocap_reference",
    points=np.zeros((n_ref_pts, 3), dtype=np.float32),
    colors=np.tile(np.array([[1.0, 0.5, 0.0]], dtype=np.float32), (n_ref_pts, 1)),
    point_size=0.025,
)

port = server.get_port()
print(f"Open in browser: http://localhost:{port}")

# Colab: embed inline
try:
    from google.colab.output import eval_js  # type: ignore
    from IPython.display import IFrame, display

    colab_url = eval_js(f"google.colab.kernel.proxyPort({port})")
    display(IFrame(src=colab_url, width="100%", height=520))
except ImportError:
    pass  # not on Colab — open the URL above manually

## 6 · Run the control loop with live visualization

Each iteration:
1. `solver.command` → optimal action from MPPI
2. `env.step` → advance the physics
3. `task.advance()` → advance the reference frame counter
4. `scene.update_from_mjdata` → push new pose to browser
5. Update orange reference cloud to the current reference frame body positions
6. `panel.update` → refresh cost / ESS / action panels

In [ ]:
import time

NUM_STEPS = 200
REALTIME = False  # False → run as fast as possible
LOG_EVERY = 50

env.reset()
total_cost = 0.0
step_start = time.perf_counter()

for step_i in range(NUM_STEPS):
    u = solver.command(env.data)
    _, reward, *_ = env.step(u)
    task.advance()  # advance reference frame — must be called after env.step()
    total_cost -= reward

    scene.update_from_mjdata(env.data)

    # Update orange reference dot cloud
    t = task._ref_idx
    ref_pelvis = task.ref_xpos_viz[t][0:1, :]
    actual_pelvis = env.data.xpos[1:2, :]
    ref_pts = (task.ref_xpos_viz[t] - ref_pelvis + actual_pelvis + scene._scene_offset).astype(np.float32)
    ref_cloud.points = ref_pts

    panel.update(
        solver.last_cost,
        solver.cost_weights,
        u,
        cost_terms=solver.last_cost_terms,
        u_nominal=solver.u_nominal_snapshot,
    )

    if REALTIME:
        elapsed = time.perf_counter() - step_start
        target = (step_i + 1) * task.dt
        if target > elapsed:
            time.sleep(target - elapsed)

    if (step_i + 1) % LOG_EVERY == 0:
        print(f"step {step_i + 1:4d}  cost={solver.last_cost:.3f}  cumulative={total_cost:.2f}")

print(f"\nDone — {NUM_STEPS} steps, total cost = {total_cost:.2f}")

## 7 · Tear down the server

In [ ]:
server.stop()

## 8 · Run the control loop

Each iteration: `solver.command` → `env.step` → `task.advance()` (advances the reference frame counter).

In [ ]:
NUM_STEPS = 200
LOG_EVERY = 50

costs_total = []
costs_body_pos = []
costs_body_ori = []
costs_config = []
costs_vel = []
pelvis_heights = []

env.reset()

for step_i in range(NUM_STEPS):
    u = solver.command(env.data)
    env.step(u)
    task.advance()  # advance reference frame — must be called after env.step()

    terms = task.cost_terms(env.data, u)
    total = sum(terms.values())

    costs_total.append(total)
    costs_body_pos.append(terms["body_position"])
    costs_body_ori.append(terms["body_orientation"])
    costs_config.append(terms["configuration"])
    costs_vel.append(terms["velocity"])
    pelvis_heights.append(float(env.data.qpos[2]))

    if (step_i + 1) % LOG_EVERY == 0:
        print(
            f"step {step_i + 1:4d}  "
            f"total={total:.4f}  "
            f"body_pos={terms['body_position']:.4f}  "
            f"body_ori={terms['body_orientation']:.4f}  "
            f"config={terms['configuration']:.4f}  "
            f"vel={terms['velocity']:.4f}  "
            f"pelvis_z={env.data.qpos[2]:.3f} m"
        )

print(f"\nDone — {NUM_STEPS} steps")

## 9 · Plot cost breakdown and pelvis height

In [ ]:
import matplotlib.pyplot as plt

steps = list(range(1, NUM_STEPS + 1))

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

ax = axes[0]
ax.plot(steps, costs_total, label="total", color="black", linewidth=2)
ax.plot(steps, costs_body_pos, label="body_position", color="#e6194b", linewidth=1.2)
ax.plot(steps, costs_body_ori, label="body_orientation", color="#f58231", linewidth=1.2)
ax.plot(steps, costs_config, label="configuration", color="#3cb44b", linewidth=1.2)
ax.plot(steps, costs_vel, label="velocity", color="#4363d8", linewidth=1.2)
ax.set_ylabel("cost (normalised)")
ax.set_title("MPPI cost term breakdown — G1 mocap tracking")
ax.legend(loc="upper right", fontsize=9)
ax.set_ylim(0, None)
ax.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.plot(steps, pelvis_heights, color="#911eb4", linewidth=1.5)
ax2.axhline(0.72, color="gray", linestyle="--", linewidth=1, label="reference ≈ 0.72 m")
ax2.set_xlabel("step")
ax2.set_ylabel("pelvis height (m)")
ax2.set_title("Pelvis height (qpos[2]) over time")
ax2.legend(loc="lower right", fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10 · Inspect per-body position errors

In [ ]:
import mujoco
import numpy as np

t = task._ref_idx  # current reference frame after the run above

actual_xipos = env.data.xipos[1:]  # (30, 3) — skip world body
ref_xipos = task.ref_xpos[t]  # (30, 3)
per_body_err = np.linalg.norm(actual_xipos - ref_xipos, axis=1)

worst_idx = np.argsort(per_body_err)[::-1][:5]
print("Top-5 bodies by position error at the last step:")
for i in worst_idx:
    bname = mujoco.mj_id2name(task.mj_model, mujoco.mjtObj.mjOBJ_BODY, i + 1)
    print(f"  {bname:30s}  err = {per_body_err[i]:.4f} m")

## Next steps

- **Try a different motion clip**: pass another `reference_filename` from the LocoMuJoCo dataset (`run1`, `dance1`, `jump1`, …).
- **Tune the solver**: increase `num_samples` to 512–1024 on a GPU; lower `temperature` to tighten tracking; increase `horizon` for more foresight.
- **Adjust cost weights**: give more weight to `body_orientation` if the robot keeps tilting, or reduce `velocity` weight if the motion looks jittery.
- **Browser visualization**: launch locally with `uv run python examples/run_mujoco_task.py g1_mocap --viewer mjviser --render` and open `http://localhost:8080`.
- **Swap the task**: replace `G1MocapTracking` with `G1VelocityTracking` for command-conditioned velocity tracking, or `HumanoidStandup` for stand-up control.